# REPA × CIFAR-10 × DINOv2

Trains **SiT-S/8** on CIFAR-10 using **DINOv2 ViT-Base** as the representation encoder.

All code runs from the cloned `nikiboura/REPA` fork.
DINOv2 downloads automatically from `torch.hub` — no extra datasets needed.

## 1. Install dependencies

In [ ]:
%%bash
pip install -q accelerate diffusers timm einops

## 2. Clone REPA fork

In [ ]:
%%bash
cd /kaggle/working
# check if folder exists already
if [ ! -d "REPA" ]; then
    git clone https://github.com/nikiboura/REPA.git REPA
fi
# latest git commit
echo "REPA ready. Commit: $(git -C REPA rev-parse --short HEAD)"

## 3. Set paths

In [ ]:
import os

# point to REPA and CIFAR-10
REPA_DIR = '/kaggle/working/REPA'
DATA_DIR = '/kaggle/working/data/cifar10_256'

# create directory for dataset
os.makedirs(DATA_DIR, exist_ok=True)

print(f'REPA dir : {REPA_DIR}')
print(f'Data dir : {DATA_DIR}')

## 4. Prepare CIFAR-10
Downloads CIFAR-10, resizes to 256×256, VAE-encodes each image, saves latents as `.npy`.

In [ ]:
import subprocess, sys, threading

cmd = [
    sys.executable, '/kaggle/working/REPA/prepare_cifar10.py',
    '--out-dir', '/kaggle/working/data/cifar10_256',
    '--resolution', '256',
    '--max-samples', '5000',
]
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

def stream(pipe):
    for line in pipe:
        print(line, end='', flush=True)

t1 = threading.Thread(target=stream, args=(process.stdout,))
t2 = threading.Thread(target=stream, args=(process.stderr,))
t1.start(); t2.start()
t1.join(); t2.join()
process.wait()
print('Exit code:', process.returncode)

## 5. Train
Runs 5000 steps as a quick correctness check. Checkpoints saved to `/kaggle/working/results/`.

**Models**
1. SiT-S/8 (Diffusion Model): denoises in latent space
2. DINOv2 ViT-Base: pretrained vision encoder (downloads automatically)

**Losses**:
1. denoising_loss: standard diffusion loss
2. proj_loss: REPA alignment loss

Total loss = denoising_loss + proj_loss * proj_coeff

In [ ]:
import subprocess, os, re
from tqdm.notebook import tqdm

env = os.environ.copy()

cmd = [
    'accelerate', 'launch',
    '--mixed_precision', 'fp16',
    '--num_processes', '1',
    '/kaggle/working/REPA/train.py',
    '--exp-name',            'repa-sits8-dinov2-cifar10',
    '--output-dir',          '/kaggle/working/results',
    '--report-to',           'none',
    '--model',               'SiT-S/8',
    '--num-classes',         '10',
    '--encoder-depth',       '8',
    '--enc-type',            'dinov2-vit-b',
    '--data-dir',            '/kaggle/working/data/cifar10_256',
    '--resolution',          '256',
    '--batch-size',          '32',
    '--num-workers',         '2',
    '--epochs',              '200',
    '--max-train-steps',     '5000',
    '--learning-rate',       '1e-4',
    '--mixed-precision',     'fp16',
    '--checkpointing-steps', '5000',
    '--proj-coeff',          '0.5',
    '--cfg-prob',            '0.1',
    '--path-type',           'linear',
]

process = subprocess.Popen(cmd, cwd='/kaggle/working/REPA', env=env,
                           stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)

pbar = tqdm(total=5000, desc='Training DINOv2')
last_step = 0
for line in process.stdout:
    m = re.search(r'Steps:\s*(\d+)/\d+.*?loss=([\d.]+).*?proj_loss=(-?[\d.]+)', line)
    if m:
        step = int(m.group(1))
        if step > last_step:
            pbar.update(step - last_step)
            pbar.set_postfix({'loss': m.group(2), 'proj': m.group(3)})
            last_step = step
pbar.close()
process.wait()
print('Training complete. Checkpoint saved.')

## 6. Generate samples

In [ ]:
import subprocess, os, glob

env = os.environ.copy()

# loads latest checkpoint
ckpts = sorted(glob.glob('/kaggle/working/results/repa-sits8-dinov2-cifar10/checkpoints/*.pt'))
print('Using checkpoint:', ckpts[-1])

cmd = [
    'torchrun', '--nproc_per_node=1',
    '/kaggle/working/REPA/generate.py',
    '--model',                'SiT-S/8',
    '--ckpt',                 ckpts[-1],
    '--encoder-depth',        '8',
    '--num-classes',          '10',
    '--projector-embed-dims', '768',
    '--per-proc-batch-size',  '16',
    '--num-fid-samples',      '64',
    '--path-type',            'linear',
    '--mode',                 'ode',
    '--num-steps',            '50',
    '--cfg-scale',            '1.0',
    '--resolution',           '256',
    '--vae',                  'mse',
]

result = subprocess.run(cmd, cwd='/kaggle/working/REPA', env=env, text=True)
print('Exit code:', result.returncode)

## 7. Show generated images

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob

npz_files = sorted(glob.glob('/kaggle/working/REPA/samples/**/*.npz', recursive=True))

# loads images
data = np.load(npz_files[-1])
imgs = data[data.files[0]]  # shape (64, 256, 256, 3)

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(imgs[i].astype('uint8'))
    ax.axis('off')
plt.tight_layout()
plt.show()
print(f'From: {npz_files[-1]}')

## 8. Compute FID score

In [ ]:
import numpy as np
import glob
from PIL import Image
from tqdm import tqdm
import subprocess

# Step 1 — build real CIFAR-10 reference .npz
real_imgs = sorted(glob.glob('/kaggle/working/data/cifar10_256/images/val/*.png'))
samples = []
for p in tqdm(real_imgs[:5000], desc='Loading real images'):
    img = np.array(Image.open(p).convert('RGB'))
    samples.append(img)
samples = np.stack(samples)
np.savez('/kaggle/working/real_cifar10.npz', arr_0=samples)
print('Saved real reference:', samples.shape)

# Step 2 — download ADM evaluator
subprocess.run(['pip', 'install', '-q', 'scipy'], check=True)
subprocess.run([
    'wget', '-q', '-O', '/kaggle/working/evaluator.py',
    'https://raw.githubusercontent.com/openai/guided-diffusion/main/evaluations/evaluator.py'
], check=True)

# Step 3 — run FID
gen_npz = sorted(glob.glob('/kaggle/working/REPA/samples/**/*.npz', recursive=True))[-1]
print('Generated npz:', gen_npz)

subprocess.run([
    'python', '/kaggle/working/evaluator.py',
    '/kaggle/working/real_cifar10.npz',
    gen_npz,
], text=True)